# Init

## Importing liabraries

In [0]:
# importing liabraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import *
from pyspark.sql import Window


## Rename config

In [0]:
# Rename config
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

# Read from the Bronze

In [0]:
df = spark.table('workspace.bronze.crm_cust_info_raw')

# Data transformation

### Triming

In [0]:
# string data triming
for field in df.schema.fields:
    if field.dataType == StringType():
        df = df.withColumn(field.name, trim(col(field.name)))


## Normalization

In [0]:
# Normalization
df = df.withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "M", "Married")
     .when(F.upper(F.col("cst_marital_status")) == "S", "Single")
     .otherwise("N/A") # Default fallback
).withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
     .when(F.upper(F.col("cst_gndr")) == "F", "Female")
     .otherwise("N/A") # Default fallback
)


## Rename column and fix column order

In [0]:
# column Rename
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
# column order
df = df.select("customer_id", "customer_key", "firstname", "lastname", "marital_status", "gender", "create_date")

## Remove duplicate and null customer ids

In [0]:
# 1. Filter out null IDs (where customer_id is not null)
df = df.filter(F.col("customer_id").isNotNull())

# 2. Define the window layout (partition by customer_id order by create_date desc)
window_spec = Window.partitionBy("customer_id").orderBy(F.col("create_date").desc())

# 3. Apply row_number and filter where row_number = 1
df = (
    df
    .withColumn("row_num", F.row_number().over(window_spec)) # Generate row numbers
    .filter(F.col("row_num") == 1)                           # Keep only latest record
    .drop("row_num")                                         # Drop the temporary column
)


## data check

In [0]:
# Data check

print("=== 1. NULL & DUPLICATE CHECKS ===")
# Count total rows, null IDs, and unique IDs
stats = df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("null_ids"),
    F.countDistinct("customer_id").alias("unique_ids")
).collect()[0]

print(f"Total Rows: {stats['total_rows']}")
print(f"Null IDs Found: {stats['null_ids']}")
print(f"Duplicate IDs Found: {stats['total_rows'] - stats['unique_ids'] - stats['null_ids']}")

print("\n=== 2. UNTRIMMED STRING CHECKS ===")
# Checks if the length of the string matches the length of the trimmed string
untrimmed_checks = df.select(
    F.sum(F.when(F.length(F.col("marital_status")) != F.length(F.trim(F.col("marital_status"))), 1).otherwise(0)).alias("untrimmed_marital"),
    F.sum(F.when(F.length(F.col("gender")) != F.length(F.trim(F.col("gender"))), 1).otherwise(0)).alias("untrimmed_gender")
).collect()[0]

print(f"Untrimmed values in marital_status: {untrimmed_checks['untrimmed_marital']}")
print(f"Untrimmed values in gender: {untrimmed_checks['untrimmed_gender']}")

print("\n=== 3. DISTINCT VALUES ===")
print("Distinct Marital Statuses:")
df.select("marital_status").distinct().show()

print("Distinct Genders:")
df.select("gender").distinct().show()


# Write in silver table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.crm_cust_info")
)

In [0]:
%sql
select * from workspace.silver.crm_cust_info